In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm                    # Для OLS-регрессии
from statsmodels.tsa.stattools import coint    # Для теста коинтеграции


In [ ]:
#Default Settings
p_value = 0.05
min_correlation = 0.5

#settings.csv
try:
    settings_df = pd.read_csv('CSV/settings.csv', sep=';', index_col='Parameter')

    if 'p_value' in settings_df.index:
        p_value = float(settings_df.loc['p_value','Value'])
        print('[S] p_value:', p_value)
    else:
        print('[D] p_value:', p_value)

    if 'min_correlation' in settings_df.index:
        min_correlation = float(settings_df.loc['min_correlation','Value'])
        print('[S] min_correlation:', min_correlation)
    else:
        print('[D] min_correlation:', min_correlation)
except Exception as e:
    print('[D] settings.csv не прочитан, дефолты: p_value=', p_value, ' min_correlation=', min_correlation)


In [ ]:
#Loading Master DF
master_df = pd.read_excel('DATA/FULL_Trading_Calendar.xlsx')
master_df['Date'] = pd.to_datetime(master_df['Date'])
master_df.set_index('Date', inplace=True)

print('[OK]')
print('Days:',master_df.shape[0])
print('Avaliable Tickers:',master_df.shape[1])

In [ ]:
#загружает расписание периодов для тестирования торговой стратегии
#подготавливает даты для работы с временными рядами.

try:
    schedule_df = pd.read_csv('CSV/schedule.csv', sep =';')

    schedule_df['IS_Start'] = pd.to_datetime(schedule_df['IS_Start'], dayfirst=True)
    schedule_df['IS_End'] = pd.to_datetime(schedule_df['IS_End'], dayfirst=True)

    schedule_df['OOS_Start'] = pd.to_datetime(schedule_df['OOS_Start'], dayfirst=True)
    schedule_df['OOS_End'] = pd.to_datetime(schedule_df['OOS_End'], dayfirst=True)

    print('[OK]')
except Exception as e:
    print('[ERROR]:', e)

In [ ]:
# номер окна (пока хардкод на первое; дальше — из счётчика цикла)
iteration_num = 1
row = schedule_df.iloc[0]

is_start = row['IS_Start']
is_end = row['IS_End']

oos_start = row['OOS_Start']
oos_end = row['OOS_End']

is_data = master_df.loc[is_start:is_end]
oos_data = master_df.loc[oos_start:oos_end]

print('[OK]')
print(f"Итерация {iteration_num}:")
print(f"  - In-Sample (окно поиска пар): {is_start.date()} -> {is_end.date()} | Рабочих дней: {len(is_data)}")
print(f"  - Out-of-Sample (окно торговли): {oos_start.date()} -> {oos_end.date()} | Рабочих дней: {len(oos_data)}")

In [ ]:
# Восстанавливаем тип актива для каждого тикера (нужно для filter_correlation)
tickers_df = pd.read_csv('CSV/Tickers.csv', keep_default_na=False)
ticker_type_map = {}
for col in tickers_df.columns:
    for ticker in tickers_df[col]:
        clean_ticker = ticker.strip().replace('.', '-') if col == 'EQUITY' else ticker.strip()
        if clean_ticker:
            ticker_type_map[clean_ticker] = col

print('[OK] Тикеров с типом:', len(ticker_type_map))

In [ ]:
def filter_correlation(is_data, ticker_type_map, min_corr=0.5):
    """
    Фильтр 1: корреляция на лог-доходностях.
    Исключает только equity-equity, остальное разрешено.
    """
    log_returns = np.log(is_data / is_data.shift(1)).dropna(how='all')
    valid_tickers = log_returns.dropna(axis=1, how='any').columns.tolist()

    results = []
    for t1, t2 in itertools.combinations(valid_tickers, 2):
        type1 = ticker_type_map.get(t1, 'UNKNOWN')
        type2 = ticker_type_map.get(t2, 'UNKNOWN')
        if type1 == 'EQUITY' and type2 == 'EQUITY':
            continue  # искл. только equity-equity

        corr = log_returns[t1].corr(log_returns[t2])
        if pd.notna(corr) and abs(corr) >= min_corr:
            results.append({
                'Asset_A': t1, 'Asset_B': t2,
                'Type_A': type1, 'Type_B': type2,
                'Pearson_Corr': round(corr, 4)
            })

    return pd.DataFrame(results)


def filter_cointegration(candidates_df, is_data, p_value=0.05):
    """
    Фильтр 2: Энгл-Грейнджер. Берёт пары из filter_correlation().
    Работает на ценах, не на доходностях.
    """
    if candidates_df.empty:
        return candidates_df

    p_values = []
    for _, row in candidates_df.iterrows():
        t1, t2 = row['Asset_A'], row['Asset_B']
        try:
            s1, s2 = is_data[t1].dropna(), is_data[t2].dropna()
            idx = s1.index.intersection(s2.index)
            _, p_val, _ = coint(s1.loc[idx], s2.loc[idx])
        except Exception:
            p_val = np.nan
        p_values.append(p_val)

    result = candidates_df.copy()
    result['P_Value'] = p_values
    result = result[result['P_Value'] <= p_value].reset_index(drop=True)
    return result

In [ ]:
# Пробный запуск на первом IS-окне (is_data уже посчитан выше)
# min_corr и p_value читаются из CSV/settings.csv
candidates = filter_correlation(is_data, ticker_type_map, min_corr=min_correlation)
candidates.insert(0, 'Iteration', iteration_num)  # номер окна

cointegrated = filter_cointegration(candidates, is_data, p_value=p_value)

print(f"После корреляции: {len(candidates)} пар | После коинтеграции: {len(cointegrated)} пар")
cointegrated.head(10)

In [ ]:
# Сохранение результата.
# Пока одно окно. После цикла по всем итерациям будет сохранять всё разом.

import os
os.makedirs('DATA', exist_ok=True)

output_path = 'DATA/is_results.xlsx'
cointegrated.to_excel(output_path, index=False)

print(f"[OK] Сохранено {len(cointegrated)} пар в {output_path}")